In [5]:
"""
Trader Decision Support — Multi-Agent Workflow
================================================
Energy Commodity Trading Commercial Desk 

Scenario: Trader asks a natural language question during a live market move.
System acts as an intelligent analyst: reads the situation, 
calculates risk, and gives a clear recommendation with reasoning.

Agents:
  - intent_agent       → understands what the trader is actually asking
  - market_agent       → interprets market conditions & price drivers
  - risk_agent         → calculates P&L exposure with tool use
  - recommendation_agent → produces a clear, trader-friendly recommendation
  - evaluator_agent    → scores quality (used offline for model improvement)

Orchestrator routes dynamically — a pure market question skips risk calc,
a position question skips market narrative, etc.
"""

import json
import re
import anthropic

with open("config.json", "r") as f:
    config = json.load(f)

API_KEY = config["API_KEY"]
MODEL = config["MODEL"]
client  = anthropic.Anthropic(api_key=API_KEY)


# ── Trader's current positions (in production: live feed from CTRM system) ────

POSITIONS = {
    "Brent Crude": {
        "direction":      "long",
        "volume_barrels": 500_000,
        "avg_cost_usd":   82.50,
        "current_price":  85.10,   # just spiked
    },
    "WTI Crude": {
        "direction":      "short",
        "volume_barrels": 200_000,
        "avg_cost_usd":   80.20,
        "current_price":  79.90,
    },
    "TTF Gas": {
        "direction":      "long",
        "volume_mwh":     150_000,
        "avg_cost_eur":   34.50,
        "current_price":  35.80,
    },
}

# ── Market context (in production: Bloomberg, Argus, Reuters feeds) ───────────

MARKET_CONTEXT = """
Date: 2024-01-15 14:32 UTC

Recent market events:
- Brent Crude up 3.1% today following Houthi attacks on Red Sea shipping lanes
- US crude inventories drew down 4.2M barrels last week (above 2.1M expected)
- Saudi Arabia signaled no production increase before Q2 review
- IEA revised 2024 demand forecast up 200k bpd citing stronger Asia demand
- USD weakened 0.4% against basket — supportive for dollar-denominated commodities
- Goldman Sachs raised Brent 3-month target from $85 to $91

Risk events this week:
- OPEC+ meeting Thursday
- US CPI data Wednesday
- EIA weekly inventory report Wednesday 15:30 UTC
"""


# ── Shared helpers ────────────────────────────────────────────────────────────

def call_claude(system, user_message, tools=None):
    kwargs = {
        "model":      MODEL,
        "max_tokens": 1500,
        "system":     system,
        "messages":   [{"role": "user", "content": user_message}],
    }
    if tools:
        kwargs["tools"] = tools
    return client.messages.create(**kwargs)

def get_text(response):
    return "\n".join(b.text for b in response.content if b.type == "text")

def get_tool_use(response):
    return next((b for b in response.content if b.type == "tool_use"), None)

def parse_json(raw):
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    return json.loads(match.group()) if match else {}

def print_agent(name, content):
    print(f"\n{'─'*65}")
    print(f"  🤖 {name}")
    print(f"{'─'*65}")
    print(content)


# ── Risk calculation tool (runs locally) ─────────────────────────────────────

PNL_TOOL = {
    "name": "calculate_pnl",
    "description": "Calculate current P&L and exposure for a trader's position.",
    "input_schema": {
        "type": "object",
        "properties": {
            "commodity": {
                "type": "string",
                "description": "e.g. 'Brent Crude', 'WTI Crude', 'TTF Gas'"
            },
            "scenario_price_usd": {
                "type": "number",
                "description": "Hypothetical price to calculate P&L against"
            }
        },
        "required": ["commodity"]
    }
}

def run_pnl_calc(commodity: str, scenario_price: float = None) -> str:
    pos = POSITIONS.get(commodity)
    if not pos:
        return f"No open position found for {commodity}."

    current = pos["current_price"]
    cost    = pos["avg_cost_usd"] if "avg_cost_usd" in pos else pos.get("avg_cost_eur", 0)
    check_price = scenario_price if scenario_price else current

    if "volume_barrels" in pos:
        volume  = pos["volume_barrels"]
        unit    = "barrels"
        pnl     = (check_price - cost) * volume
        if pos["direction"] == "short":
            pnl = -pnl
    else:
        volume  = pos["volume_mwh"]
        unit    = "MWh"
        pnl     = (check_price - cost) * volume

    label = "Scenario" if scenario_price else "Current"
    return (
        f"{commodity} | {pos['direction'].upper()} {volume:,} {unit}\n"
        f"  Cost: {cost:.2f} | {label} price: {check_price:.2f}\n"
        f"  Unrealised P&L: ${pnl:,.0f}"
    )


# ── Specialist Agents ─────────────────────────────────────────────────────────

def intent_agent(query: str) -> dict:
    """Understands what the trader is actually asking."""
    response = call_claude(
        system="""You are an intent classifier for a commodity trading desk.
Classify the trader's query into:
- query_type: one of [position_risk, market_outlook, hedge_decision, scenario_analysis, general]
- commodities: list of commodities mentioned or implied
- urgency: high / medium / low
- needs_pnl_calc: true/false — does this need a P&L calculation?
- needs_market_context: true/false — does this need market narrative?

Return JSON only: {
  "query_type": "...",
  "commodities": [...],
  "urgency": "...",
  "needs_pnl_calc": true,
  "needs_market_context": true,
  "summary": "one line summary of what trader wants"
}""",
        user_message=query
    )
    result = parse_json(get_text(response))
    print_agent("Intent Agent", json.dumps(result, indent=2))
    return result


def market_agent(query: str, commodities: list) -> str:
    """Interprets current market conditions relevant to the query."""
    response = call_claude(
        system="""You are a senior commodity market analyst at a trading house.
You have access to current market data and news.
Give a sharp, concise market read — the way a good analyst briefs a trader.
3-5 bullet points max. No waffle. Focus on what matters for the trade decision.""",
        user_message=f"""Trader query: {query}

Commodities of interest: {', '.join(commodities)}

Current market context:
{MARKET_CONTEXT}

Give a brief market read relevant to this query."""
    )
    result = get_text(response)
    print_agent("Market Agent", result)
    return result


def risk_agent(query: str, commodities: list) -> str:
    """Calculates P&L exposure using the tool."""
    # Ask Claude which commodity and scenario to calculate
    response = call_claude(
        system="You are a risk analyst. Use the calculate_pnl tool to assess P&L exposure for the trader's query. Calculate for each relevant commodity.",
        user_message=f"""Trader query: {query}

Open positions available: {', '.join(POSITIONS.keys())}
Relevant commodities: {', '.join(commodities)}

Calculate current P&L and if a price move is implied, calculate the scenario too.""",
        tools=[PNL_TOOL]
    )

    results = []
    # Handle multiple tool calls
    for block in response.content:
        if block.type == "tool_use":
            commodity       = block.input.get("commodity")
            scenario_price  = block.input.get("scenario_price_usd")
            results.append(run_pnl_calc(commodity, scenario_price))

    if not results:
        results = [run_pnl_calc(c) for c in commodities if c in POSITIONS]

    output = "\n\n".join(results) if results else "No matching positions found."
    print_agent("Risk Agent", output)
    return output


def recommendation_agent(query: str, context: dict) -> str:
    """Produces a clear, trader-friendly recommendation."""
    context_str = ""
    if context.get("market_read"):
        context_str += f"Market context:\n{context['market_read']}\n\n"
    if context.get("pnl"):
        context_str += f"Position & P&L:\n{context['pnl']}\n\n"

    response = call_claude(
        system="""You are a senior trader's analyst at a major commodity trading house.
Give a direct, confident recommendation.

Format your response exactly like this:
RECOMMENDATION: [BUY HEDGE / HOLD / REDUCE / ADD] 
CONVICTION: [High / Medium / Low]

RATIONALE (2-3 lines):
...

KEY RISKS:
- ...
- ...

SUGGESTED ACTION (specific and actionable):
...

Be direct. Traders don't want essays — they want clarity under pressure.""",
        user_message=f"""Trader's question: {query}

{context_str}

Give your recommendation."""
    )
    result = get_text(response)
    print_agent("Recommendation Agent", result)
    return result


def evaluator_agent(query: str, recommendation: str) -> dict:
    """Scores the recommendation — used offline to improve the system."""
    response = call_claude(
        system="""You are a senior trading desk evaluator reviewing AI-generated recommendations.
Score on:
- relevance: did it answer what the trader actually asked? (1-5)
- actionability: is the recommendation specific and executable? (1-5)  
- risk_awareness: did it flag the key risks? (1-5)
- clarity: would a busy trader understand this in 10 seconds? (1-5)

Return JSON only: {
  "relevance": N,
  "actionability": N,
  "risk_awareness": N,
  "clarity": N,
  "overall": N,
  "improvement": "one specific thing that would make this better"
}""",
        user_message=f"Trader query: {query}\n\nRecommendation:\n{recommendation}"
    )
    result = parse_json(get_text(response))
    print_agent("Evaluator Agent", json.dumps(result, indent=2))
    return result


# ── Dynamic Router ────────────────────────────────────────────────────────────

def route(intent: dict) -> list:
    """
    Build the agent pipeline from the intent classification.
    No Claude call needed here — the intent agent already gave us
    everything we need to make the routing decision in pure Python.
    """
    pipeline = []

    if intent.get("needs_market_context"):
        pipeline.append("market")

    if intent.get("needs_pnl_calc"):
        pipeline.append("risk")

    pipeline.append("recommendation")   # always
    pipeline.append("evaluator")        # always

    return pipeline


# ── Orchestrator ──────────────────────────────────────────────────────────────

def orchestrator(trader_query: str) -> dict:
    print(f"\n{'═'*65}")
    print(f"  TRADER QUERY: {trader_query}")
    print(f"{'═'*65}")

    context = {}

    # Step 1: Always classify intent first
    intent = intent_agent(trader_query)
    context["intent"] = intent
    commodities = intent.get("commodities", ["Brent Crude"])

    # Step 2: Decide pipeline from intent
    pipeline = route(intent)
    print(f"\n  📋 Pipeline: intent → {' → '.join(pipeline)}")

    # Step 3: Run each agent
    for agent in pipeline:
        if agent == "market":
            context["market_read"] = market_agent(trader_query, commodities)

        elif agent == "risk":
            context["pnl"] = risk_agent(trader_query, commodities)

        elif agent == "recommendation":
            context["recommendation"] = recommendation_agent(trader_query, context)

        elif agent == "evaluator":
            context["eval"] = evaluator_agent(trader_query, context["recommendation"])

    print(f"\n{'═'*65}")
    print(f"  DONE — Eval score: {context['eval'].get('overall')}/5")
    print(f"{'═'*65}\n")

    return context


# ── Evaluation suite ──────────────────────────────────────────────────────────

def run_eval_suite():
    """
    Offline evaluation across different query types.
    In production: log all queries + scores to a database,
    track score trends over time, flag regressions.
    """
    test_cases = [
        {
            "query": "Brent just spiked 3%. We're long 500k barrels. Should I hedge?",
            "expected_agents": ["market", "risk", "recommendation", "evaluator"],
            "expected_query_type": "hedge_decision",
        },
        {
            "query": "What's driving TTF gas today?",
            "expected_agents": ["market", "recommendation", "evaluator"],
            "expected_query_type": "market_outlook",
        },
        {
            "query": "Show me my current P&L on WTI.",
            "expected_agents": ["risk", "recommendation", "evaluator"],
            "expected_query_type": "position_risk",
        },
        {
            "query": "If Brent drops to $78, what does that do to our book?",
            "expected_agents": ["market", "risk", "recommendation", "evaluator"],
            "expected_query_type": "scenario_analysis",
        },
    ]

    results = []

    for i, tc in enumerate(test_cases, 1):
        print(f"\n{'#'*65}")
        print(f"  TEST CASE {i}/{len(test_cases)}")
        print(f"{'#'*65}")

        output = orchestrator(tc["query"])

        # Check routing was correct
        intent          = output["intent"]
        actual_type     = intent.get("query_type")
        expected_type   = tc["expected_query_type"]
        routing_correct = actual_type == expected_type

        eval_scores = output.get("eval", {})

        result = {
            "query":            tc["query"],
            "routing_correct":  routing_correct,
            "expected_type":    expected_type,
            "actual_type":      actual_type,
            "eval_scores":      eval_scores,
            "overall":          eval_scores.get("overall"),
            "improvement":      eval_scores.get("improvement"),
        }
        results.append(result)

    # Summary
    print(f"\n{'═'*65}")
    print("  EVAL SUMMARY")
    print(f"{'═'*65}")

    routing_acc = sum(r["routing_correct"] for r in results) / len(results) * 100
    avg_score   = sum(r["overall"] for r in results if r["overall"]) / len(results)

    print(f"  Routing accuracy : {routing_acc:.0f}%")
    print(f"  Avg overall score: {avg_score:.1f}/5")
    print(f"  Avg relevance    : {sum(r['eval_scores'].get('relevance',0) for r in results)/len(results):.1f}/5")
    print(f"  Avg actionability: {sum(r['eval_scores'].get('actionability',0) for r in results)/len(results):.1f}/5")
    print(f"  Avg clarity      : {sum(r['eval_scores'].get('clarity',0) for r in results)/len(results):.1f}/5")

    print(f"\n  Areas to improve:")
    for r in results:
        if r["improvement"]:
            print(f"  • [{r['actual_type']}] {r['improvement']}")

    return results


# ── Run ───────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Single query
    orchestrator("Brent just spiked 3%. We're long 500k barrels. Should I hedge?")

    # Full eval suite
    # run_eval_suite()


═════════════════════════════════════════════════════════════════
  TRADER QUERY: Brent just spiked 3%. We're long 500k barrels. Should I hedge?
═════════════════════════════════════════════════════════════════

─────────────────────────────────────────────────────────────────
  🤖 Intent Agent
─────────────────────────────────────────────────────────────────
{
  "query_type": "hedge_decision",
  "commodities": [
    "Brent Crude"
  ],
  "urgency": "high",
  "needs_pnl_calc": true,
  "needs_market_context": true,
  "summary": "Trader seeks hedging guidance on a 500k barrel long Brent position following a sudden 3% price spike."
}

  📋 Pipeline: intent → market → risk → recommendation → evaluator

─────────────────────────────────────────────────────────────────
  🤖 Market Agent
─────────────────────────────────────────────────────────────────
## Brent Market Read — 2024-01-15 14:32 UTC

**Fundamentals are stacked bullish — but you're sitting on a 3% gain with catalysts still ahead.**

